# Horizon-Aware Dynamic Control of Residual Alpha with Jump--Decay and Heavy Tails

## Overview

This framework describes a discrete-time trading controller for short-horizon residual alpha in a small equity universe. The objective is to: choose both a position and a trading horizon while accounting for (1) factor exposure, (2) microstructure signals, (3) latent alpha persistence, (4) jumps, (5) heavy-tailed noise, (6) transaction costs, and (7) regime uncertainty. 

The motivating setting is a medium-frequency strategy operating on a fixed decision grid (we start with every $15$ seconds), using high-frequency quote and trade data aggregated into features. The controller is designed to distinguish between signals that matter at very short horizons (order-flow imbalance and spread pressure), and signals that matter over longer horizons (factor drift, cross-asset flows, and persistent residual mispricing).

## Dynamic programming formulation

Let decision times be indexed by $t=0,1,2,\ldots$, with one decision step equal to $\Delta t$ units of clock time. In the first experiments, I take $\Delta t=15$ seconds, so the controller observes new data and re-optimizes every $15$ seconds. The traded assets are Apple, Microsoft, and NVIDIA. The return, factor return, and position vectors are respectively

$$
r_t =
\begin{pmatrix}
r_{\mathrm{AAPL},t} \\
r_{\mathrm{MSFT},t} \\
r_{\mathrm{NVDA},t}
\end{pmatrix}
\in \mathbb{R}^3,
\quad
f_t =
\begin{pmatrix}
r_{\mathrm{SPY},t} \\
r_{\mathrm{QQQ},t}
\end{pmatrix}
\in \mathbb{R}^2,
\quad
q_t =
\begin{pmatrix}
q_{\mathrm{AAPL},t} \\
q_{\mathrm{MSFT},t} \\
q_{\mathrm{NVDA},t}
\end{pmatrix}
\in \mathbb{R}^3.
$$
where SPY proxies for broad market movement and QQQ proxies for large-cap technology and Nasdaq-related movement. 

In the position vector ${q_t}$, positive entries represent long exposure and negative entries represent short exposure. In our initial implementation, $q_t$ is restricted to a finite set of candidate positions repreenting unit exposures, and so the action set is

$$
\mathcal{Q}
=
\{(a,b,c):a,b,c\in\{-1,0,1\}\}
$$

where $a,b,c$ denote binary exposures to AAPL, MSFT, and NVDA respectively. Equivalently, the strategy can choose no trade, a single-name long or short position, or a market-neutral pair trade between any two of Apple, Microsoft, and NVIDIA. This discrete action space is useful because it makes the control problem transparent and reduces the risk of overfitting continuous position sizes. It also allows the strategy to compare directional trades against relative-value trades (e.g., long AAPL and short MSFT).

At time $t$, the controller observes an information set $\mathcal{F}_t$ containing past returns, quote features, trade features, factor returns, factor loadings, cost estimates, posterior regime probabilities, and posterior samples of latent alpha. The state is written as

$$
S_t =
\left(
q_t,
\mathcal{P}_t(Z_t,s_t),
B_t,
\Sigma_t,
C_t,
X_t
\right),
$$

where $q_t$ is the current position, $\mathcal{P}_t(Z_t,s_t)$ is the posterior distribution of the latent residual alpha $Z_t$ and regime $s_t$ given $\mathcal{F}_t$, $B_t$ is the matrix of factor loadings, $\Sigma_t$ is the return covariance estimate, $C_t$ is the transaction cost environment, and $X_t$ is the vector of microstructure features. The latent variable $Z_t \in \mathbb{R}^3$ represents persistent residual alpha after removing factor-driven movement. The regime variable $s_t \in \{1,\ldots,K\}$ represents the current latent market state, such as a high-liquidity mean-reversion state, a momentum/order-flow state, a volatile jump state, or a risk-off state.

The fully dynamic problem can be written using the Bellman equation

$$
V(S_t)
=
\max_{q'_t \in \mathcal{Q}}
\left\{
R(S_t,q'_t)
+
\mathbb{E}
\left[
V(S_{t+1})
\mid
S_t,q'_t
\right]
\right\},
$$

where $V(S_t)$ is the value of being in state $S_t$, $q'_t$ is the next chosen position, and $R(S_t,q'_t)$ is the immediate reward net of costs. In principle, this is the correct dynamic programming formulation. In practice, direct solution is infeasible because the state contains continuous posterior beliefs, latent variables, regime uncertainty, time-varying covariance, nonlinear costs, jumps, and heavy-tailed noise. Therefore, the strategy uses a tractable finite-horizon surrogate. At each decision time, the controller evaluates a finite set of candidate horizons

$$
\mathcal{H}=\{1,2,4,8,20\},
$$

where these values are measured in decision steps. With $\Delta t=15$ seconds, this corresponds to horizons of $15$ seconds, $30$ seconds, $1$ minute, $2$ minutes, and $5$ minutes. The controller then chooses the position-horizon pair that maximizes an estimated risk-adjusted value. This creates a receding-horizon control rule: execute the chosen position now, observe new data at the next decision time, update beliefs, and re-optimize.

## Factor-adjusted residual alpha

The return model separates factor movement from residual alpha. Over one decision interval, returns obey

$$
r_t
=
B_t f_t
+
A_t \Delta t
+
\epsilon_t,
$$

where $B_t \in \mathbb{R}^{3 \times 2}$ is the time-varying loading matrix on SPY and QQQ, $A_t \in \mathbb{R}^3$ is the instantaneous residual alpha vector, and $\epsilon_t \in \mathbb{R}^3$ is idiosyncratic noise. The purpose of this decomposition is to avoid confusing broad market movement with tradable residual signal. For example, if Apple, Microsoft, and NVIDIA rise because QQQ rises, that is not necessarily stock-specific alpha. The residual component is intended to capture what remains after accounting for broad market and technology-sector movement.

The feature vector $X_t$ is constructed from sub-second quote and trade data and then aggregated to the decision grid. The variables we are using  are (FOR NOW - STILL WORKING ON THIS!!) bid-ask spread, quoted depth, order-flow imbalance, trade imbalance, signed volume, realized volatility, short-lag returns, quote revisions, cross-asset return spreads, rolling correlation, and time-of-day variables. These features are allowed to matter differently across horizons. At very short horizons, the relevant signal mostly come from liquidity pressure, imbalance, and temporary order-flow shocks. At longer horizons, the relevant signal come more from factor drift, cross-asset flows, and persistent residual movements.

To capture this, residual alpha is modeled as horizon-aware. For each candidate horizon $n \in \mathcal{H}$, define

$$
A_t(n)
=
w_{\mathrm{micro}}(n) f_{\mathrm{micro}}(X_t)
+
w_{\mathrm{factor}}(n) g(f_t,\Delta f_t)
+
Z_t.
$$

Here, $f_{\mathrm{micro}}:\mathbb{R}^{d_X}\to\mathbb{R}^3$ maps microstructure features into a fast alpha signal for AAPL, MSFT, and NVDA, while $g:\mathbb{R}^{d_f}\to\mathbb{R}^3$ maps factor-related variables into a slower residual alpha signal. The term $\Delta f_t$ can include short-window changes in SPY and QQQ returns, factor momentum, factor volatility, or cross-factor spreads. The weights $w_{\mathrm{micro}}(n)$ and $w_{\mathrm{factor}}(n)$ determine how much each signal component matters at horizon $n$. They satisfy

$$
w_{\mathrm{micro}}(n) \geq 0,
\qquad
w_{\mathrm{factor}}(n) \geq 0,
\qquad
w_{\mathrm{micro}}(n)+w_{\mathrm{factor}}(n)=1.
$$

The key restriction is monotonicity:

$$
w_{\mathrm{micro}}(n) \downarrow \text{ as } n \uparrow,
\qquad
w_{\mathrm{factor}}(n) \uparrow \text{ as } n \uparrow.
$$

This means the model explicitly encodes the idea that microstructure pressure should decay quickly, while slower factor or cross-asset information may remain relevant for longer. For now, we use a simple parametric choice

$$
w_{\mathrm{micro}}(n)
=
\exp(-\kappa_w n),
\qquad
w_{\mathrm{factor}}(n)
=
1-\exp(-\kappa_w n),
$$

where $\kappa_w>0$ controls how quickly the strategy transitions from microstructure-dominated signals to factor-dominated signals. (we could instead treat weights as hyper-parameters and the weights can be estimated or tuned on a validation set.)

## Latent alpha (w/ jump-decay dynamics)

The persistent component of residual alpha follows a regime-dependent jump--decay process. The latent alpha evolves according to

$$
Z_t
=
R^{(s_t)} Z_{t-1}
+
J_t
+
u_t,
$$

where $R^{(s_t)}\in\mathbb{R}^{3\times 3}$ is the regime-dependent persistence matrix, $J_t\in\mathbb{R}^3$ is a jump term, and $u_t\in\mathbb{R}^3$ is a heavy-tailed innovation. The matrix $R^{(s_t)}$ controls how quickly residual alpha decays. Its diagonal terms describe own-asset persistence, while its off-diagonal terms allow cross-asset spillovers. For example, an NVIDIA residual shock may partially predict later residual movement in Apple or Microsoft if the three names are linked by common technology-sector flows, AI-related news, semiconductor demand, or broader growth-stock sentiment.

The state innovation is modeled as heavy-tailed:

$$
u_t \sim t_{\nu_Z}(0,Q^{(s_t)}),
$$

where $t_{\nu_Z}$ denotes a multivariate Student-$t$ distribution with degrees of freedom $\nu_Z$ and scale matrix $Q^{(s_t)}\in\mathbb{R}^{3\times 3}$. Smaller values of $\nu_Z$ imply fatter tails. This is more realistic than Gaussian noise at short horizons because intraday returns and residuals often contain outliers, bursts of volatility, and non-normal shocks.

The jump term is

$$
J_t
=
N_t \odot Y_t,
$$

where $\odot$ denotes elementwise multiplication. For asset $i\in\{\mathrm{AAPL},\mathrm{MSFT},\mathrm{NVDA}\}$,

$$
N_{i,t}
\sim
\mathrm{Bernoulli}(\pi_{i,t}^{(s_t)}),
\qquad
Y_{i,t}
\sim
H_i^{(s_t)}.
$$

Here, $N_{i,t}$ determines whether a jump occurs in asset $i$, $\pi_{i,t}^{(s_t)}$ is the regime-dependent jump intensity, and $Y_{i,t}$ is the jump size drawn from distribution $H_i^{(s_t)}$. The jump intensity depends on current features,

$$
\pi_{i,t}^{(k)}
=
\Lambda_i^{(k)}(X_t),
$$

 ($\Lambda_i^{(k)}$ can be a logistic model, tree model, NN, or other probabilistic classifier - we could try a few different ones and see what works best). This allows the probability of a jump to increase during periods with wide spreads, abnormal order-flow bursts, sudden quote revisions, elevated volatility, or specific times of day such as the open and close.

The observation equation is also heavy-tailed. Since

$$
r_t
=
B_t f_t
+
A_t \Delta t
+
\epsilon_t,
$$

a one-step factor-adjusted residual observation can be written as

$$
\widetilde{y}_t
=
\frac{r_t-B_t f_t}{\Delta t}
-
f_{\mathrm{micro}}(X_t)
-
g(f_t,\Delta f_t).
$$

This residual observation isolates the part of returns not explained by observed factors and the directly observed alpha features. The measurement equation is

$$
\widetilde{y}_t
=
Z_t
+
\xi_t,
$$

where

$$
\xi_t \sim t_{\nu_\xi}(0,R_\xi),
\qquad
R_\xi \in \mathbb{R}^{3\times 3}.
$$

The Student-$t$ observation noise makes the filter less sensitive to extreme residual observations. This matters because a Gaussian filter may overreact to isolated outliers, while a heavy-tailed likelihood can treat unusual observations as possible noise rather than immediately revising the latent alpha too aggressively.

## Regime uncertainty

Regimes are introduced by allowing the parameters of the model to change across latent states. The regime process satisfies

$$
s_t \in \{1,\ldots,K\},
$$

and evolves according to a Markov transition matrix

$$
\Pi_{ij}
=
\Pr(s_t=j \mid s_{t-1}=i).
$$

At time $t$, the controller maintains posterior regime probabilities

$$
p_{k,t}
=
\Pr(s_t=k \mid \mathcal{F}_t),
\qquad
k=1,\ldots,K.
$$

Each regime has its own parameter collection,

$$
\Theta^{(k)}
=
\left\{
R^{(k)},
Q^{(k)},
\Sigma^{(k)},
C^{(k)},
\pi^{(k)},
H^{(k)}
\right\}.
$$

This means that alpha persistence, covariance, costs, jump probabilities, and jump-size distributions can differ across market states. For example, in a calm mean-reverting regime, $R^{(k)}$ may imply fast decay and low jump intensity. In a momentum regime, $R^{(k)}$ may imply stronger persistence. In a volatile regime, $Q^{(k)}$, $\Sigma^{(k)}$, and $\pi^{(k)}$ may all be larger.

Because the model contains jumps, heavy tails, and regimes, a standard Kalman filter is not sufficient. The posterior distribution

$$
p(Z_t,s_t \mid \mathcal{F}_t)
$$

is approximated using a particle filter. Let the particle system be

$$
\left\{
Z_t^{(m)},s_t^{(m)},w_t^{(m)}
\right\}_{m=1}^M,
$$

where $m$ indexes particles, $Z_t^{(m)}\in\mathbb{R}^3$ is the latent alpha sample, $s_t^{(m)}$ is the regime sample, and $w_t^{(m)}$ is the particle weight. The prediction step samples a new regime from the Markov transition matrix,

$$
s_t^{(m)}
\sim
\Pi_{s_{t-1}^{(m)},\cdot},
$$

then draws jumps according to the regime-specific jump model and propagates latent alpha using

$$
Z_t^{(m)}
=
R^{(s_t^{(m)})}Z_{t-1}^{(m)}
+
J_t^{(m)}
+
u_t^{(m)}.
$$

The update step weights each particle by the Student-$t$ likelihood of the residual observation,

$$
w_t^{(m)}
\propto
w_{t-1}^{(m)}
\,
p(\widetilde{y}_t \mid Z_t^{(m)},s_t^{(m)}).
$$

The weights are normalized so that

$$
\sum_{m=1}^{M} w_t^{(m)}=1.
$$

The effective sample size is

$$
\mathrm{ESS}_t
=
\frac{1}{\sum_{m=1}^{M}(w_t^{(m)})^2}.
$$

When $\mathrm{ESS}_t$ falls below a threshold, such as $M/2$, the particles are resampled. This prevents the filter from collapsing onto a small number of particles.

For each candidate horizon $n$, posterior alpha samples are computed as

$$
A_t^{(m)}(n)
=
w_{\mathrm{micro}}(n) f_{\mathrm{micro}}(X_t)
+
w_{\mathrm{factor}}(n) g(f_t,\Delta f_t)
+
Z_t^{(m)}.
$$

The posterior mean alpha is

$$
\widehat{A}_t(n)
=
\sum_{m=1}^{M}
w_t^{(m)} A_t^{(m)}(n).
$$

Tail functionals such as value-at-risk and conditional value-at-risk are also computed directly from the particle-implied distribution rather than imposed through a Gaussian approximation.

## Horizon-dependent objects (decay, risk, and costs)

The persistence matrix can be connected to continuous-time exponential decay. For each regime $k$, define a decay matrix $K^{(k)}$ such that

$$
R^{(k)}
=
\exp(-K^{(k)}\Delta t).
$$

If the eigenvalues of $R^{(k)}$ are close to one, residual alpha decays slowly and remains valuable over longer horizons. If the eigenvalues are close to zero, alpha decays quickly and the controller should prefer shorter horizons. Conditional on regime $k$ and current alpha $a\in\mathbb{R}^3$, the expected cumulative alpha over $n$ steps is

$$
G^{(k)}(n)a
=
\sum_{\ell=1}^{n}
(R^{(k)})^\ell a
=
\left(
\sum_{\ell=1}^{n}
(R^{(k)})^\ell
\right)a.
$$

When $I-R^{(k)}$ is invertible, this matrix sum can be written as

$$
G^{(k)}(n)
=
R^{(k)}
\left[
I-(R^{(k)})^n
\right]
\left[
I-R^{(k)}
\right]^{-1}.
$$

This expression makes the horizon decision interpretable. More persistent alpha increases the cumulative expected return from holding longer, while faster decay makes longer horizons less attractive.

Risk is also horizon-dependent. Let

$$
\Sigma_t^{(k)}(n)
\approx
\operatorname{Var}^{(k)}(r_{t+1}+\cdots+r_{t+n}\mid \mathcal{F}_t)
$$

denote the regime-specific covariance of cumulative returns over horizon $n$, where $\Sigma_t^{(k)}(n)\in\mathbb{R}^{3\times 3}$. Let

$$
S_t^{(k)}
=
\operatorname{Var}(Z_t \mid s_t=k,\mathcal{F}_t)
$$

denote posterior uncertainty about latent alpha within regime $k$, estimated from particles. A combined risk matrix can be written as

$$
\Omega_t^{(k)}(n)
=
\lambda_\Sigma \Sigma_t^{(k)}(n)
+
\psi S_t^{(k)},
$$

where $\lambda_\Sigma>0$ weights return risk and $\psi>0$ weights estimation uncertainty about alpha. This separates ordinary price risk from uncertainty about whether the signal itself is reliable.

Transaction costs are modeled as a convex function of turnover. If the controller moves from current position $q_t$ to candidate position $q$, then

$$
\Delta q_t
=
q-q_t.
$$

The cost function is

$$
C_t(\Delta q_t)
=
\sum_{i=1}^{3}
\left[
\frac{\mathrm{spread}_{i,t}}{2}
|\Delta q_{i,t}|
+
\eta_i |\Delta q_{i,t}|^\gamma
\right],
\qquad
\gamma>1.
$$

The first term approximates half-spread crossing costs, while the second term captures nonlinear market impact or slippage. The coefficient $\eta_i>0$ controls the severity of nonlinear execution costs for asset $i$. 

(Not sure if this would add too much instability) The cost function can also be regime-dependent,

$$
C_t^{(k)}(\Delta q_t),
$$

because spreads, liquidity, and impact are typically worse in volatile regimes.

## Control rule and regime belief aggregation

For a candidate position $q\in\mathbb{R}^3$ and horizon $n$, define cumulative PnL as

$$
\Pi_{t,n}(q)
=
\sum_{\ell=1}^{n}
q^\top A_{t+\ell}(n)\Delta t.
$$

In empirical implementation, the distribution of $\Pi_{t,n}(q)$ is approximated using posterior particles and regime beliefs. The controller uses a tail-aware objective,

$$
J_t(n,q)
=
\mathbb{E}_t[\Pi_{t,n}(q)]
-
\lambda_{\mathrm{CVaR}}
\operatorname{CVaR}_{\alpha}
\left(
-\Pi_{t,n}(q)
\right)
-
C_t(q-q_t),
$$

where $\lambda_{\mathrm{CVaR}}>0$ is the tail-risk aversion parameter and $\operatorname{CVaR}_{\alpha}(-\Pi_{t,n}(q))$ is the expected loss conditional on being in the worst $\alpha$ tail of the loss distribution. This objective differs from a standard mean-variance objective because it penalizes downside tail losses rather than only variance. This is important when residual returns are heavy-tailed and jump-prone.

The control rule is

$$
(q_t^\ast,n_t^\ast)
=
\arg\max_{q\in\mathcal{Q},\,n\in\mathcal{H}}
J_t(n,q).
$$

The controller executes $q_t^\ast$ and then repeats the entire process at the next decision time. This is a receding-horizon strategy: the model does not commit to holding the position for the full selected horizon regardless of new information. Instead, the horizon $n_t^\ast$ is interpreted as the horizon over which the current signal is most valuable, while the actual position is updated dynamically as new observations arrive.

Because the true regime $s_t$ is unobserved, the controller must decide how to aggregate regime-specific objectives. Let

$$
J_t^{(k)}(n,q)
$$

be the value of choosing horizon $n$ and position $q$ under regime $k$. I.e.,

$$
J_t^{(k)}(n,q)
=
\mathbb{E}_t^{(k)}[\Pi_{t,n}(q)]
-
\lambda_{\mathrm{CVaR}}
\operatorname{CVaR}_{\alpha}^{(k)}
\left(
-\Pi_{t,n}(q)
\right)
-
C_t^{(k)}(q-q_t).
$$

The most conservative Bayesian rule averages regime-specific objectives using posterior regime probabilities:

$$
J_t^{\mathrm{linear}}(n,q)
=
\sum_{k=1}^{K}
p_{k,t}J_t^{(k)}(n,q).
$$

This rule is theoretically clean because each regime contributes according to its posterior probability. However, it can dilute strong signals when the posterior is diffuse. For example, if one regime strongly favors a trade but the filter assigns only moderate probability to that regime, the averaged objective may be too weak to overcome transaction costs.

To reduce dilution while preserving some uncertainty, define sharpened regime weights

$$
\widetilde{p}_{k,t}(\rho)
=
\frac{p_{k,t}^{\rho}}
{\sum_{j=1}^{K}p_{j,t}^{\rho}},
\qquad
\rho>1.
$$

The sharpened objective is

$$
J_t^{\mathrm{sharp}}(n,q;\rho)
=
\sum_{k=1}^{K}
\widetilde{p}_{k,t}(\rho)J_t^{(k)}(n,q).
$$

The parameter $\rho$ controls how aggressively the controller emphasizes the most likely regime. When $\rho=1$, sharpened weighting reduces to linear weighting. As $\rho\to\infty$, the rule approaches hard regime selection. This makes $\rho$ a useful robustness parameter. A natural empirical grid is

$$
\rho \in \{1,1.5,2,4,8\}.
$$

The third rule is maximum-probability regime selection. Define

$$
k_t^\ast
=
\arg\max_{k\in\{1,\ldots,K\}}
p_{k,t}.
$$

Then

$$
J_t^{\mathrm{max}}(n,q)
=
J_t^{(k_t^\ast)}(n,q).
$$

This rule avoids signal dilution entirely because the controller acts as if the most likely regime is the true regime. It may perform well when regime classification is accurate, but it is fragile when posterior probabilities are close. Small changes in the belief vector can switch the selected regime, potentially causing discontinuous trading decisions and excess turnover.

All three aggregation rules can be written using a generic regime weight vector

$$
\omega_t
=
(\omega_{1,t},\ldots,\omega_{K,t}),
\qquad
\sum_{k=1}^{K}\omega_{k,t}=1,
\qquad
\omega_{k,t}\geq 0.
$$

The aggregated objective is

$$
J_t^{\omega}(n,q)
=
\sum_{k=1}^{K}
\omega_{k,t}J_t^{(k)}(n,q).
$$

The final robust control rule is

$$
(q_t^\ast,n_t^\ast)
=
\arg\max_{q\in\mathcal{Q},\,n\in\mathcal{H}}
J_t^{\omega}(n,q).
$$

The three choices of $\omega_t$ are

$$
\omega_{k,t}^{\mathrm{linear}}
=
p_{k,t},
$$

$$
\omega_{k,t}^{\mathrm{sharp}}
=
\frac{p_{k,t}^{\rho}}
{\sum_{j=1}^{K}p_{j,t}^{\rho}},
$$

and

$$
\omega_{k,t}^{\mathrm{max}}
=
\mathbf{1}\{k=k_t^\ast\}.
$$

In summary, linear weighting should be the most stable and conservative. Maximum-probability weighting should be the most decisive but also the most sensitive to classification error. Sharpened weighting is the compromise between the two.



## Assumptions. 

(1) factor separability assumes that SPY and QQQ capture enough market and sector movement that the remaining residual component contains potentially tradable alpha. (2) horizon decomposition assumes that microstructure signals decay faster than factor or cross-asset signals. (3) jump-decay dynamics assume that latent residual alpha is persistent but mean-reverting, with occasional discontinuous shocks. (4) heavy-tailed noise assumes that both state innovations and observations contain more extreme values than a Gaussian model would allow. (5) regime dependence assumes that market behavior is piecewise stable, with different parameters in different latent states. (6) convex transaction costs assume that larger trades are increasingly expensive because of spread, slippage, and market impact.

The comparative statics are intuitive. If the microstructure signal $f_{\mathrm{micro}}(X_t)$ becomes stronger relative to the slower factor signal $g(f_t,\Delta f_t)$, the optimal horizon $n_t^\ast$ should shift shorter because the source of alpha is expected to decay quickly. If the slower factor component becomes stronger, the optimal horizon should shift longer. If the eigenvalues of $R^{(k)}$ move closer to one, latent alpha becomes more persistent, the cumulative gain matrix $G^{(k)}(n)$ increases, and longer horizons become more attractive. If jump intensity $\pi_{i,t}^{(k)}$ increases, tail risk rises, so the CVaR penalty increases and the controller should reduce position size or shorten the horizon unless jumps are directionally favorable. If the degrees of freedom $\nu_Z$ or $\nu_\xi$ decrease, tails become fatter, which also raises the tail-risk penalty and makes the strategy more conservative. If volatility or posterior alpha uncertainty increases, the risk matrix $\Omega_t^{(k)}(n)$ rises and the controller should either trade less or require a stronger expected alpha to justify the position. If cross-asset correlation rises, same-direction positions become riskier because $q^\top \Sigma q$ increases, while spread positions such as $(1,-1,0)$, $(1,0,-1)$, or $(0,1,-1)$ may become relatively more attractive. Finally, higher spreads and slippage increase $C_t(q-q_t)$, creating stronger inertia and requiring larger expected returns before trading.

Empirically, the model should be evaluated using out-of-sample performance after realistic costs. Important metrics include mean return, volatility, Sharpe ratio, maximum drawdown, hit rate, turnover, average holding horizon, average spread paid, realized slippage, tail losses, and performance by regime. It is also useful to evaluate performance conditional on posterior regime entropy,

$$
H(p_t)
=
-\sum_{k=1}^{K}
p_{k,t}\log p_{k,t}.
$$

When entropy is low, the regime posterior is concentrated and all aggregation rules should behave similarly. When entropy is high, the posterior is diffuse and the choice between linear, sharpened, and maximum-probability aggregation matters more. If hard regime selection performs well only when entropy is low, then maximum-probability control should be used cautiously. If sharpened weighting improves performance during moderate-entropy periods, then it may provide the best practical balance between uncertainty smoothing and decisive trading.

The main limitations are also clear. The horizon weights $w_{\mathrm{micro}}(n)$ and $w_{\mathrm{factor}}(n)$ may be misspecified, which can cause the model to assign short-term signals to long horizons or long-term signals to short horizons. The jump model may be wrong if jump intensities or jump-size distributions are poorly estimated. Regime misclassification can distort persistence, covariance, cost, and jump estimates. Particle filtering can be computationally expensive, especially if the model is updated every few seconds. Finally, the execution model is still simplified because true profitability also depends on queue position, latency, fill probability, order type, and the difference between backtested prices and executable prices.

## Interpretation

The main interpretation is that the controller removes broad factor movement using SPY and QQQ, constructs horizon-aware residual alpha from fast microstructure signals and slower factor-related signals, tracks persistent alpha using a jump--decay latent process, updates beliefs through a particle filter, penalizes downside tail risk using CVaR, and chooses both the trade and the horizon through a finite action search. This makes the strategy more realistic than a simple return-forecasting rule because the model explicitly asks which signal matters, how long it should last, how uncertain the regime is, how costly the trade is, and how severe the downside tail risk may be.

## References